In [0]:
data = [
    (1, "John", 50000),
    (2, "David", 60000),
    (3, "Smith", 45000),
    (4, "Alice", 75000)
]

columns = ["emp_id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

In [0]:
df.filter(df.salary > 50000).show()

In [0]:
df.select("emp_id", "name").show()

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("bonus", col("salary") * 0.10)

In [0]:
df = df.withColumnRenamed("name", "employee_name").show()

In [0]:
from pyspark.sql.functions import col

# Cell 5 overwrote df with None (`.show()` returns None), so rebuild df here
data = [
    (1, "John", 50000),
    (2, "David", 60000),
    (3, "Smith", 45000),
    (4, "Alice", 75000)
]

df = spark.createDataFrame(data, ["emp_id", "name", "salary"]) \
    .withColumn("bonus", col("salary") * 0.10) \
    .withColumnRenamed("name", "employee_name")

df.select("salary") \
  .distinct() \
  .orderBy(col("salary").desc()) \
  .limit(2) \
  .show()

In [0]:
df.groupBy("emp_id") \
  .count() \
  .filter(col("count") > 1) \
  .show()

In [0]:
df.dropDuplicates(["emp_id"]).show()

In [0]:
data = [
    (1, "John", "IT", 50000),
    (2, "David", "HR", 60000),
    (3, "Smith", "IT", 70000),
    (4, "Alice", "HR", 80000)
]

df = spark.createDataFrame(
    data, ["emp_id", "name", "department", "salary"]
)

In [0]:
from pyspark.sql.functions import avg

df.groupBy("department") \
  .agg(avg("salary").alias("avg_salary")) \
  .show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

w = Window.partitionBy("department")

df = df.withColumn(
    "dept_avg_salary",
    avg("salary").over(w)
)

df.filter(
    col("salary") > col("dept_avg_salary")
).show()

In [0]:
#advanced pyspark
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

w = Window.partitionBy("department") \
          .orderBy(col("salary").desc())

result = df.withColumn(
    "rank",
    row_number().over(w)
)

result.filter(col("rank") <= 2).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

w = Window.orderBy("emp_id") \
          .rowsBetween(
              Window.unboundedPreceding,
              Window.currentRow
          )

df.withColumn(
    "cumulative_salary",
    sum("salary").over(w)
).show()

In [0]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
df.fillna({
    "name": "Unknown",
    "salary": 0
}).show()

In [0]:
emp_data = [
    (1, "John", 101),
    (2, "David", 102),
    (3, "Smith", 103)
]

dept_data = [
    (101, "IT"),
    (102, "HR"),
    (103, "Finance")
]

emp_df = spark.createDataFrame(
    emp_data, ["emp_id", "name", "dept_id"]
)

dept_df = spark.createDataFrame(
    dept_data, ["dept_id", "department"]
)

result = emp_df.join(dept_df, "dept_id", "inner")

result.show()